# MOABB Multi-Dataset MI Fine-Tuning with SignalJEPA (Within-Subject 5-Fold)

Configurable notebook for moabb datasets. It follows SignalJEPA fine-tuning workflow: average reference, resample to 128 Hz, bandpass 0.5-40 Hz, event-window creation, post-window microvolt scaling, S-JEPA downstream model loading, and artifact saving.


# 1. Setup


In [ ]:
import os
os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import random
import sys
from pathlib import Path
import platform
import json
import hashlib
from datetime import datetime
import math

import numpy as np
from numpy import multiply

import torch
from torch.utils.data import Subset

from braindecode import EEGClassifier
from braindecode.models import SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual
from braindecode.datasets import MOABBDataset, BaseConcatDataset
from braindecode.datautil import load_concat_dataset
from braindecode.preprocessing import (
    Preprocessor,
    preprocess,
    create_windows_from_events,
)

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, confusion_matrix
from skorch.callbacks import EarlyStopping
from skorch.dataset import ValidSplit

import builtins

import pandas as pd
import matplotlib.pyplot as plt

import mne
mne.set_log_level("WARNING")

print("All imports loaded successfully.")

In [ ]:
print("Runtime Environment:")
print(f"  - Python: {sys.version}")
print(f"  - Platform: {platform.platform()}")

WORKING_DIR = Path.cwd().parent
print(f"\nWorking directory: {WORKING_DIR}")


# 2. Configuration


## 2.1 Default Dataset Settings

In [ ]:
# Dataset defaults for MOABB motor-imagery transfer experiments.
MOABB_MI_DATASET_CONFIGS = {
    "Lee2019_MI": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Lee2019_MI.html
        "goal": "same-dataset reproduction for Lee2019-pretrained S-JEPA",
        # "default_subjects": list(range(48, 55)),
        "default_subjects": list(range(48, 55)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
        "base_trial_duration_s": 4.0,
        "target_window_duration_s": 4.2,
    },
    "Cho2017": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Cho2017.html
        "goal": "closest external high-density binary left/right MI transfer test",
        # "default_subjects": list(range(1, 53)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
        "base_trial_duration_s": 3.0,
        "target_window_duration_s": 3.2,
    },
    "Stieger2021": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Stieger2021.html
        "goal": "same 62-channel scale, more sessions, left/right subset only",
        # "default_subjects": list(range(1, 63)),
        "default_subjects": list(range(1, 6)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
        "base_trial_duration_s": 3.0,
        "target_window_duration_s": 3.2,
    },
    "PhysionetMI": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.PhysionetMI.html
        "goal": "hard cross-dataset stress test",
        # "default_subjects": list(range(1, 110)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [88],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {"imagined": True, "executed": False},
        "base_trial_duration_s": 3.0,
        "target_window_duration_s": 3.2,
    },
    "BNCI2014_001": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.BNCI2014_001.html
        "goal": "standard BCI-IV-2a benchmark, left/right subset only",
        # "default_subjects": list(range(1, 10)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
        "base_trial_duration_s": 4.0,
        "target_window_duration_s": 4.2,
    },
    "Weibo2014": {
        # https://moabb.neurotechx.com/docs/generated/moabb.datasets.Weibo2014.html
        "goal": "broader benchmark comparison, left/right subset only",
        # "default_subjects": list(range(1, 11)),
        "default_subjects": list(range(1, 10)),
        "exclude_subjects": [],
        "labels_to_keep": ["left_hand", "right_hand"],
        "moabb_dataset_kwargs": {},
        "base_trial_duration_s": 8,
        "target_window_duration_s": 8.2,
    },
}

## 2.2 CONFIG

In [ ]:
CONFIG = {
    # Paths
    "artifact_dir": str(WORKING_DIR / "artifacts" / "moabb-mi-sjepa"),

    # Dataset source
    # - "moabb": build through braindecode.datasets.MOABBDataset
    # - "concat": load an already saved Braindecode concat dataset from preprocessed_concat_dir
    "dataset_source": "moabb",
    "dataset_name": "Stieger2021",  # Lee2019_MI, Cho2017, Stieger2021, PhysionetMI, BNCI2014_001, Weibo2014
    "preprocessed_concat_dir": None,

    # Set these to None to use MOABB_MI_DATASET_CONFIGS defaults.
    "labels_to_keep": None,
    "exclude_subjects": None,
    "subjects_to_use": None,
    "moabb_dataset_kwargs": None,
    "target_window_duration_s": None,

    # Reproducibility
    "seed": 12,
    "set_seed": True,

    # Preprocessing
    "sfreq": 128,
    "bandpass_low": 0.5,
    "bandpass_high": 40.0,

    # Model settings
    "model_name": "SignalJEPA_PreLocal",  # SignalJEPA_PreLocal, SignalJEPA_PostLocal, SignalJEPA_Contextual
    "pretrained_mode": "from_pretrained", # from_pretrained or random

    # Strategy settings
    "strategy": "new",  # new, full
    "warmup_epochs": 10,

    # Cross-validation
    "cv_folds": 5,
    "val_split": 0.2,

    # Training
    "batch_size": 32,
    "n_epochs": 50,
    "early_stopping_patience": None,
    "learning_rate": 0.001,

    # PreLocal spatial-filter interpretability
    # Only applies to SignalJEPA_PreLocal because PreLocal uses spatial_conv before the local encoder.
    "extract_spatial_conv_weights": True,
    "save_spatial_weight_plots": True,
    "plot_individual_spatial_filters": True,
    "max_spatial_filters_to_plot": 8,
    "topomap_dpi": 160,

    # Diagnostics for failure-mode analysis.
    # These are especially useful for PhysioNet/Weibo where the classifier can collapse to one class.
    "collapse_threshold": 0.90,
    "log_spatial_update_stats": True,
    "log_probability_diagnostics": True,
}

In [ ]:
if CONFIG["dataset_name"] not in MOABB_MI_DATASET_CONFIGS:
    raise ValueError(
        f"Unsupported dataset_name={CONFIG['dataset_name']}. "
        f"Supported: {sorted(MOABB_MI_DATASET_CONFIGS)}"
    )

_DATASET_DEFAULTS = MOABB_MI_DATASET_CONFIGS[CONFIG["dataset_name"]]

# Fill defaults into CONFIG so config.json contains the effective run settings.
if CONFIG["labels_to_keep"] is None:
    CONFIG["labels_to_keep"] = list(_DATASET_DEFAULTS["labels_to_keep"])
if CONFIG["exclude_subjects"] is None:
    CONFIG["exclude_subjects"] = list(_DATASET_DEFAULTS["exclude_subjects"])
if CONFIG["moabb_dataset_kwargs"] is None:
    CONFIG["moabb_dataset_kwargs"] = dict(_DATASET_DEFAULTS["moabb_dataset_kwargs"])
if CONFIG["target_window_duration_s"] is None:
    CONFIG["target_window_duration_s"] = float(_DATASET_DEFAULTS["target_window_duration_s"])

DEFAULT_SUBJECTS = [
    int(s) for s in _DATASET_DEFAULTS["default_subjects"]
    if int(s) not in set(int(x) for x in CONFIG["exclude_subjects"])
]

CLASSES_MAPPING = {label: idx for idx, label in enumerate(CONFIG["labels_to_keep"])}
TARGET_N_CLASSES = len(CLASSES_MAPPING)
BASE_TRIAL_DURATION_S = float(_DATASET_DEFAULTS["base_trial_duration_s"])
TARGET_TRIAL_DURATION_S = float(CONFIG["target_window_duration_s"])

print("Selected dataset defaults:")
print(f"  Dataset:                 {CONFIG['dataset_name']}")
print(f"  Goal:                    {_DATASET_DEFAULTS['goal']}")
print(f"  Labels:                  {CONFIG['labels_to_keep']}")
print(f"  Class mapping:           {CLASSES_MAPPING}")
print(f"  Excluded subjects:       {CONFIG['exclude_subjects']}")
print(f"  Dataset kwargs:          {CONFIG['moabb_dataset_kwargs']}")
print(f"  Target window duration:  {TARGET_TRIAL_DURATION_S}")


## 2.3 Artifact Creation and Logging Init

In [ ]:
def create_run_id():
    timestamp = datetime.now().strftime("%Y%m%d_%H%M")
    config_str = json.dumps(CONFIG, sort_keys=True, default=str)
    config_hash = hashlib.md5(config_str.encode()).hexdigest()[:8]
    return f"{timestamp}_{config_hash}"

RUN_ID = create_run_id()
ARTIFACT_DIR = Path(CONFIG["artifact_dir"]) / CONFIG["dataset_name"] / RUN_ID
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

LOG_PATH = ARTIFACT_DIR / "run.log"
_LOG_FILE_HANDLE = open(LOG_PATH, "a", buffering=1, encoding="utf-8", errors="replace")

def _safe_write_text(stream, text):
    try:
        stream.write(text)
        return
    except UnicodeEncodeError:
        pass

    # Fallback for Windows terminals using a limited code page.
    encoding = getattr(stream, "encoding", None) or "utf-8"
    safe_text = text.encode(encoding, errors="replace").decode(encoding, errors="replace")
    stream.write(safe_text)

def _timestamped_print(*args, **kwargs):
    sep = kwargs.pop("sep", " ")
    end = kwargs.pop("end", "\n")
    flush = kwargs.pop("flush", False)
    file = kwargs.pop("file", None)
    message = sep.join(str(arg) for arg in args)
    leading_newlines = len(message) - len(message.lstrip("\n"))
    message_body = message[leading_newlines:]

    def _write_target(text):
        if file is None:
            _safe_write_text(sys.stdout, text)
            if flush:
                sys.stdout.flush()
        else:
            _safe_write_text(file, text)
            if flush and hasattr(file, "flush"):
                file.flush()

    if leading_newlines > 0:
        blanks = "\n" * leading_newlines
        _write_target(blanks)
        _safe_write_text(_LOG_FILE_HANDLE, blanks)

    if message_body:
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        stamped = f"[{ts}] {message_body}"
        _write_target(stamped + end)
        _safe_write_text(_LOG_FILE_HANDLE, stamped + end)
    else:
        _write_target(end)
        _safe_write_text(_LOG_FILE_HANDLE, end)

    if flush:
        _LOG_FILE_HANDLE.flush()

builtins.print = _timestamped_print


## 2.4 Reproducibility

In [ ]:
def resolve_device():
    if torch.backends.mps.is_available() and torch.backends.mps.is_built():
        return torch.device("mps")
    if torch.cuda.is_available():
        return torch.device("cuda")
    return torch.device("cpu")

DEVICE = resolve_device()
print(f"Using device: {DEVICE}")

def seed_everything(seed: int):
    os.environ["PYTHONHASHSEED"] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.benchmark = False
        torch.backends.cudnn.deterministic = True
    torch.use_deterministic_algorithms(True, warn_only=True)

BASE_SEED = int(CONFIG["seed"]) if CONFIG["seed"] is not None else None
if CONFIG["set_seed"]:
    if BASE_SEED is None:
        raise ValueError("Seed control is enabled but CONFIG['seed'] is None.")
    seed_everything(BASE_SEED)
    print(f"Seed initialized: {BASE_SEED}")
else:
    print("Seed control disabled")

print("=" * 70)
print("CONFIGURATION")
print("=" * 70)
for key in sorted(CONFIG.keys()):
    print(f"  {key}: {CONFIG[key]}")
print("=" * 70)

config_path = ARTIFACT_DIR / "config.json"
with open(config_path, "w") as f:
    json.dump(CONFIG, f, indent=2, default=str)
print(f"Config saved to: {config_path}")


# 3. Load and Prepare Data


## 3.1 Derived Values

In [ ]:
def compute_epoch_window(sfreq, target_trial_duration_s):
    sfreq = float(sfreq)
    target_trial_duration_s = float(target_trial_duration_s)
    window_size_samples = int(math.floor(target_trial_duration_s * sfreq))
    print("Epoch window configuration:")
    print(f"  Target sfreq:            {sfreq:.0f} Hz")
    print(f"  Requested duration:      {target_trial_duration_s:.3f} s")
    print(f"  Expected window samples: {window_size_samples}")
    return window_size_samples

WINDOW_SAMPLES = compute_epoch_window(CONFIG["sfreq"], TARGET_TRIAL_DURATION_S)

SUBJECTS = CONFIG["subjects_to_use"] if CONFIG["subjects_to_use"] is not None else DEFAULT_SUBJECTS
SUBJECTS = [int(s) for s in SUBJECTS]
invalid_subjects = sorted(set(SUBJECTS).intersection(set(CONFIG["exclude_subjects"])))
if invalid_subjects:
    raise ValueError(f"These subject IDs are excluded by config and cannot be loaded: {invalid_subjects}")

label_fragment = "_".join(CONFIG["labels_to_keep"])
label_hash = hashlib.md5(label_fragment.encode()).hexdigest()[:8]
subject_fragment = "_".join(str(s) for s in SUBJECTS)
subject_hash = hashlib.md5(subject_fragment.encode()).hexdigest()[:8]
window_hash = hashlib.md5(str(TARGET_TRIAL_DURATION_S).encode()).hexdigest()[:6]

if CONFIG["preprocessed_concat_dir"] is None:
    PREPROCESSED_CONCAT_DATASETS_NAME = (
        f"{CONFIG['dataset_name']}_{label_hash}_w{window_hash}_subjects_{subject_hash}"
    )
    PREPROCESSED_CONCAT_DIR = WORKING_DIR / "preprocessed_concat_datasets" / PREPROCESSED_CONCAT_DATASETS_NAME
else:
    PREPROCESSED_CONCAT_DIR = Path(CONFIG["preprocessed_concat_dir"])

PREPROCESSED_CONCAT_DATASETS_EXISTS = PREPROCESSED_CONCAT_DIR.exists() and PREPROCESSED_CONCAT_DIR.is_dir()
print(f"\nPreprocessed concat dataset exists: {PREPROCESSED_CONCAT_DATASETS_EXISTS} at {PREPROCESSED_CONCAT_DIR}")

## 3.2 Download Data

In [ ]:
if CONFIG["dataset_source"] not in {"moabb", "concat"}:
    raise ValueError("CONFIG['dataset_source'] must be either 'moabb' or 'concat'.")

if CONFIG["dataset_source"] == "concat" and not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    raise FileNotFoundError("CONFIG['dataset_source'] is 'concat' but preprocessed_concat_dir does not exist.")

if CONFIG["dataset_source"] == "moabb" and not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    print(f"\nBuilding MOABBDataset for {CONFIG['dataset_name']}...")
    print(f"  Subject IDs:     {SUBJECTS}")
    print(f"  Dataset kwargs:  {CONFIG['moabb_dataset_kwargs']}")
    DATASET = MOABBDataset(
        dataset_name=CONFIG["dataset_name"],
        subject_ids=SUBJECTS,
        dataset_kwargs=dict(CONFIG["moabb_dataset_kwargs"]),
    )
    unique_labels = sorted({
        str(desc)
        for ds in DATASET.datasets
        for desc in np.asarray(ds.raw.annotations.description).astype(str)
    })
    print(f"  Recordings loaded: {len(DATASET.datasets)}")
    print(f"  Raw labels:        {unique_labels}")
else:
    print("\nUsing existing preprocessed concat dataset.")


## 3.3 Preprocess Data

In [ ]:
def scale_volts_to_microvolts(data):
    return multiply(data, 1e6)

if CONFIG["dataset_source"] == "moabb" and not PREPROCESSED_CONCAT_DATASETS_EXISTS:
    print("Applying preprocessing...")
    preprocess(
        DATASET,
        [
            Preprocessor("pick_types", eeg=True, meg=False, stim=False),
            Preprocessor("set_eeg_reference", ref_channels="average"),
            Preprocessor("resample", sfreq=CONFIG["sfreq"]),
            Preprocessor("filter", l_freq=CONFIG["bandpass_low"], h_freq=CONFIG["bandpass_high"]),
            Preprocessor(scale_volts_to_microvolts),
        ],
        n_jobs=1,
    )
    DATASET.save(PREPROCESSED_CONCAT_DIR, overwrite=True)
    print(f"Saved preprocessed raw concat dataset to: {PREPROCESSED_CONCAT_DIR}")

## 3.4 Load the preprocessed concat dataset

In [ ]:
LOADED_DATASET = load_concat_dataset(
    PREPROCESSED_CONCAT_DIR,
    preload=True,
    target_name=None,
    ids_to_load=None,
)
CHS_INFO = LOADED_DATASET.datasets[0].raw.info["chs"]
first_raw = LOADED_DATASET.datasets[0].raw
CH_NAMES = list(first_raw.ch_names)
EEG_INFO = first_raw.info.copy()
all_annotation_labels = sorted({
    str(desc)
    for ds in LOADED_DATASET.datasets
    for desc in np.asarray(ds.raw.annotations.description).astype(str)
})
subject_ids_loaded = sorted(set(str(ds.description["subject"]) for ds in LOADED_DATASET.datasets), key=lambda x: int(x) if x.isdigit() else x)
print("\nPost-load raw validation")
print(f"  Recordings:          {len(LOADED_DATASET.datasets)}")
print(f"  Subjects loaded:     {subject_ids_loaded}")
print(f"  sfreq first rec:     {first_raw.info['sfreq']} Hz")
print(f"  EEG channel count:   {len(first_raw.ch_names)}")
print(f"  Labels present:      {all_annotation_labels}")
print(f"  Labels selected:     {CONFIG['labels_to_keep']}")

## 3.5 Create Windows

In [ ]:
def summarize_annotation_counts(dataset):
    counts = {}
    for ds in dataset.datasets:
        descriptions = np.asarray(ds.raw.annotations.description).astype(str)
        for desc in descriptions:
            counts[desc] = counts.get(desc, 0) + 1
    return dict(sorted(counts.items()))

def filter_annotations_by_description(dataset, descriptions_to_keep):
    keep_set = set(str(d) for d in descriptions_to_keep)
    total_before = 0
    total_after = 0
    for ds in dataset.datasets:
        annotations = ds.raw.annotations
        if len(annotations) == 0:
            continue
        descriptions = np.asarray(annotations.description).astype(str)
        mask = np.isin(descriptions, list(keep_set))
        total_before += len(annotations)
        total_after += int(np.sum(mask))
        filtered_annotations = mne.Annotations(
            onset=np.asarray(annotations.onset)[mask],
            duration=np.asarray(annotations.duration)[mask],
            description=descriptions[mask].tolist(),
            orig_time=annotations.orig_time,
        )
        ds.raw.set_annotations(filtered_annotations)
    return total_before, total_after

def update_annotation_durations(dataset, target_duration_s):
    updated = 0
    keep = set(CONFIG["labels_to_keep"])
    for ds in dataset.datasets:
        annotations = ds.raw.annotations
        if len(annotations) == 0:
            continue
        descriptions = np.asarray(annotations.description).astype(str)
        mask = np.isin(descriptions, list(keep))
        if not np.any(mask):
            continue
        durations = np.asarray(annotations.duration, dtype=float).copy()
        durations[mask] = float(target_duration_s)
        annotations.duration = durations
        updated += int(np.sum(mask))
    return updated

def build_windows_dataset(dataset, target_duration_s, mapping):
    before_counts = summarize_annotation_counts(dataset)
    print(f"Annotation counts before filtering: {before_counts}")

    total_before, total_after = filter_annotations_by_description(dataset, list(mapping.keys()))
    print(f"Filtered annotations: kept {total_after} / {total_before}")

    after_counts = summarize_annotation_counts(dataset)
    print(f"Annotation counts after filtering:  {after_counts}")

    updated_count = update_annotation_durations(dataset, target_duration_s=target_duration_s)
    print(f"Updated annotation durations:       {updated_count}")

    datasets_with_events = [ds for ds in dataset.datasets if len(ds.raw.annotations) > 0]
    dropped_recordings = len(dataset.datasets) - len(datasets_with_events)
    if dropped_recordings > 0:
        print(f"Dropped recordings with zero kept events: {dropped_recordings}")
    if len(datasets_with_events) == 0:
        raise RuntimeError("No recordings contain the selected labels after filtering.")

    filtered_dataset = BaseConcatDataset(datasets_with_events)
    windows_dataset = create_windows_from_events(
        filtered_dataset,
        trial_start_offset_samples=0,
        mapping=mapping,
        window_size_samples=None,
        window_stride_samples=None,
        drop_last_window=False,
        preload=True,
    )
    return windows_dataset

WINDOWS_DATASET = build_windows_dataset(
    LOADED_DATASET,
    target_duration_s=TARGET_TRIAL_DURATION_S,
    mapping=CLASSES_MAPPING,
)

In [ ]:
sample0 = WINDOWS_DATASET[0]
x0, y0 = sample0[0], sample0[1]
print(f"Window shape: {tuple(x0.shape)}")
print(f"Window sample length expected={WINDOW_SAMPLES}, got={x0.shape[-1]}")
print(f"Class mapping: {CLASSES_MAPPING}")
ALL_LABELS = np.asarray([int(WINDOWS_DATASET[i][1]) for i in range(len(WINDOWS_DATASET))], dtype=np.int64)
print(f"Global class counts: {np.bincount(ALL_LABELS, minlength=TARGET_N_CLASSES).tolist()}")
print(f"Chance level: {1.0 / TARGET_N_CLASSES:.2f}")

## 3.6 Create Subject Windows

In [ ]:
def _sort_subject_key(x):
    sx = str(x)
    return int(sx) if sx.isdigit() else sx

def get_subject_windows(windows_dataset, subjects):
    target_subjects = {str(s) for s in subjects}
    subject_to_indices = {str(s): [] for s in subjects}
    for idx, ds in enumerate(windows_dataset.datasets):
        subject_id = str(ds.description.get("subject"))
        if subject_id in target_subjects:
            subject_to_indices[subject_id].append(idx)

    missing_subjects = [sid for sid, idxs in subject_to_indices.items() if len(idxs) == 0]
    if missing_subjects:
        print(f"Subjects with no windows after filtering: {missing_subjects}")

    subject_windows = {}
    for sid in sorted(subject_to_indices.keys(), key=_sort_subject_key):
        idxs = subject_to_indices[sid]
        if len(idxs) == 0:
            continue
        subject_windows[sid] = BaseConcatDataset([windows_dataset.datasets[i] for i in idxs])
    if len(subject_windows) == 0:
        raise RuntimeError("No subject windows were created.")
    return subject_windows

SUBJECT_WINDOWS = get_subject_windows(WINDOWS_DATASET, SUBJECTS)

def summarize_subject_windows(subject_windows, n_classes):
    print("Summarizing per-subject windows...")
    for subject_id in sorted(subject_windows, key=_sort_subject_key):
        ds = subject_windows[subject_id]
        y = np.asarray([int(ds[i][1]) for i in range(len(ds))], dtype=np.int64)
        counts = np.bincount(y, minlength=n_classes)
        x = ds[0][0]
        print(f"  Subject {subject_id}: n_windows={len(ds)}, window_shape={tuple(x.shape)}, class_counts={counts.tolist()}")

summarize_subject_windows(SUBJECT_WINDOWS, TARGET_N_CLASSES)


# 4. Model


## 4.1 Build Model

In [ ]:
MODEL_REGISTRY = {
    "SignalJEPA_PreLocal": SignalJEPA_PreLocal,
    "SignalJEPA_PostLocal": SignalJEPA_PostLocal,
    "SignalJEPA_Contextual": SignalJEPA_Contextual,
}

DEFAULT_PRETRAINED_REPOS = {
    "SignalJEPA_Contextual": "braindecode/signal-jepa",
    "SignalJEPA_PostLocal": "braindecode/signal-jepa_without-chans",
    "SignalJEPA_PreLocal": "braindecode/signal-jepa_without-chans",
}

def get_model_class(model_name):
    if model_name not in MODEL_REGISTRY:
        raise ValueError(f"Unsupported model_name {model_name}; supported={list(MODEL_REGISTRY)}")
    return MODEL_REGISTRY[model_name]

def build_model(model_name):
    model_cls = get_model_class(model_name)
    n_chans = len(CHS_INFO)
    n_times = WINDOW_SAMPLES
    mode = CONFIG["pretrained_mode"]
    repo_id = DEFAULT_PRETRAINED_REPOS[model_name]
    if mode == "from_pretrained":
        model = model_cls.from_pretrained(
            repo_id,
            n_chans=n_chans,
            n_times=n_times,
            n_outputs=TARGET_N_CLASSES,
            strict=False,
        )
        info = {"loading_path": "from_pretrained", "repo_id": repo_id, "mode": mode}
    elif mode == "random":
        model = model_cls(n_chans=n_chans, n_times=n_times, n_outputs=TARGET_N_CLASSES)
        info = {"loading_path": "random_initialization", "repo_id": None, "mode": mode}
    else:
        raise ValueError("CONFIG['pretrained_mode'] must be 'from_pretrained' or 'random'.")
    return model, info

PRETRAINED_CHECKPOINT_INFO = {}

def initialize_model(model_name):
    model, info = build_model(model_name)
    info["model_name"] = model_name
    return model, info

In [ ]:
# Verify once that the selected model builds.
test_model, test_info = build_model(CONFIG["model_name"])
print(f"{CONFIG['model_name']} instantiated successfully.")
print(f"  Loading mode:          {test_info['loading_path']}")
print(f"  Repo:                  {test_info.get('repo_id')}")
print(f"  Total parameters:      {sum(p.numel() for p in test_model.parameters()):,}")
print(f"  Trainable parameters:  {sum(p.numel() for p in test_model.parameters() if p.requires_grad):,}")
print(test_model)
del test_model

## 4.2 Trainable Parameter Phases

In [ ]:
NEW_LAYER_PREFIXES = {
    "SignalJEPA_PreLocal": ("spatial_conv.", "final_layer."),
    "SignalJEPA_PostLocal": ("final_layer.",),
    "SignalJEPA_Contextual": ("pos_encoder.", "final_layer."),
}

def count_total_and_trainable_params(model):
    total = sum(p.numel() for p in model.parameters())
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    return total, trainable

def get_new_layer_prefixes(model_name):
    if model_name not in NEW_LAYER_PREFIXES:
        raise ValueError(f"Unsupported model_name for trainable phase logic: {model_name}")
    return NEW_LAYER_PREFIXES[model_name]

def set_trainable_params_for_phase(model, model_name, phase):
    if phase not in ("new", "warmup", "full"):
        raise ValueError(f"Unsupported phase: {phase}")
    trainable_names = []
    if phase == "full":
        for _, param in model.named_parameters():
            param.requires_grad = True
        trainable_names = [name for name, p in model.named_parameters() if p.requires_grad]
        phase_groups = ["all_parameters"]
    else:
        downstream_prefixes = get_new_layer_prefixes(model_name)
        for _, param in model.named_parameters():
            param.requires_grad = False
        for name, param in model.named_parameters():
            if any(name.startswith(prefix) for prefix in downstream_prefixes):
                param.requires_grad = True
                trainable_names.append(name)
        phase_groups = list(downstream_prefixes)

    total, trainable = count_total_and_trainable_params(model)
    if trainable == 0:
        raise RuntimeError(f"No trainable parameters found for model={model_name}, phase={phase}.")
    return {
        "model_name": model_name,
        "phase": phase,
        "trainable_groups": phase_groups,
        "total_params": int(total),
        "trainable_params": int(trainable),
        "trainable_ratio": float(trainable / total),
        "trainable_names": trainable_names,
    }

def assert_expected_trainable_scope(summary, model_name, phase):
    if phase == "full":
        return
    allowed_prefixes = get_new_layer_prefixes(model_name)
    unexpected_names = [name for name in summary["trainable_names"] if not any(name.startswith(prefix) for prefix in allowed_prefixes)]
    if unexpected_names:
        raise RuntimeError(f"Unexpected trainable parameters for {model_name} phase={phase}: {unexpected_names[:10]}")

def describe_trainable_params(summary, max_names=12):
    print(f"      Trainable groups: {summary['trainable_groups']}")
    print(f"      Total params:     {summary['total_params']:,}")
    print(f"      Trainable params: {summary['trainable_params']:,}")
    print(f"      Trainable pct:    {100.0 * summary['trainable_ratio']:.2f}%")
    if len(summary["trainable_names"]) <= max_names:
        preview_names = summary["trainable_names"]
    else:
        preview_names = summary["trainable_names"][:max_names]
    print(f"      Trainable parameter names: {preview_names}")
    if len(preview_names) < len(summary["trainable_names"]):
        print(f"      ... {len(summary['trainable_names']) - len(preview_names)} additional trainable parameters hidden")


def summarize_trainable_parameters(model):
    """Return a JSON-safe list of all parameters and whether they are trainable."""
    module = model.module_ if hasattr(model, "module_") else model
    rows = []
    for name, param in module.named_parameters():
        rows.append({
            "name": str(name),
            "shape": list(param.shape),
            "num_params": int(param.numel()),
            "requires_grad": bool(param.requires_grad),
        })
    return rows

def count_trainable_from_rows(rows):
    return int(sum(row["num_params"] for row in rows if row["requires_grad"]))


## 4.3 PreLocal Spatial Convolution Weight Helpers

These helpers extract the learned `spatial_conv` filters after each fold. For `SignalJEPA_PreLocal`, this is the main interpretable spatial layer because the contextual transformer attention layers are not used in the PreLocal downstream forward pass.


In [ ]:
def _json_safe_float_list(values, decimals=8):
    arr = np.asarray(values, dtype=float)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    return np.round(arr, decimals=decimals).tolist()

def _json_safe_float(value, decimals=8):
    if value is None:
        return None
    value = float(np.nan_to_num(value, nan=0.0, posinf=0.0, neginf=0.0))
    return round(value, decimals)

def get_model_module(model_or_clf):
    """Support either a raw torch module or a skorch/EEGClassifier object."""
    return model_or_clf.module_ if hasattr(model_or_clf, "module_") else model_or_clf

def _find_first_spatial_weight(model_or_clf):
    """Return (parameter_name, detached_weight_tensor) for the main PreLocal spatial_conv weight."""
    model = get_model_module(model_or_clf)
    if not hasattr(model, "spatial_conv"):
        return None, None

    # Prefer named parameters under spatial_conv that look like weights.
    candidates = []
    for name, param in model.named_parameters():
        if name.startswith("spatial_conv.") and name.endswith("weight"):
            if param.ndim >= 2:
                candidates.append((name, param.detach().cpu().clone()))
    if candidates:
        # In Braindecode SignalJEPA_PreLocal this is usually spatial_conv.1.weight.
        candidates = sorted(candidates, key=lambda x: (0 if "spatial_conv.1.weight" in x[0] else 1, x[0]))
        return candidates[0]

    # Fallback: search modules inside spatial_conv.
    for name, module in model.spatial_conv.named_modules():
        if hasattr(module, "weight") and module.weight is not None and module.weight.ndim >= 2:
            return f"spatial_conv.{name}.weight", module.weight.detach().cpu().clone()

    return None, None

def _spatial_weight_to_channel_matrix(weight_tensor, n_chans):
    """Convert Conv/Linear spatial weight tensor into [n_virtual_filters, n_input_channels]."""
    w = weight_tensor.detach().cpu().float().numpy()
    if w.ndim == 2:
        mat = w.copy()
    else:
        # Common case: Conv2d weight [out_filters, in_filters, n_chans, 1] -> [out_filters, n_chans].
        mat = w.reshape(w.shape[0], -1)

    if mat.shape[1] == n_chans:
        pass
    elif mat.shape[0] == n_chans:
        mat = mat.T
    elif mat.shape[1] % n_chans == 0:
        # If extra singleton/filter dimensions were flattened with channels, average over them.
        mat = mat.reshape(mat.shape[0], -1, n_chans).mean(axis=1)
    elif mat.size % n_chans == 0:
        mat = mat.reshape(-1, n_chans)
    else:
        raise ValueError(
            f"Cannot reshape spatial_conv weight of shape {tuple(w.shape)} into channel matrix with n_chans={n_chans}."
        )
    return mat

def get_spatial_conv_weight_matrix(model_or_clf, ch_names):
    """
    Return (weight_matrix, parameter_name) where weight_matrix is [n_virtual_filters, n_channels].
    Use this before and after training to verify that spatial_conv actually changed.
    """
    param_name, weight_tensor = _find_first_spatial_weight(model_or_clf)
    if weight_tensor is None:
        raise RuntimeError("Could not find spatial_conv weight on model.")
    weight_matrix = _spatial_weight_to_channel_matrix(weight_tensor, n_chans=len(ch_names))
    return weight_matrix.copy(), param_name

def compute_spatial_update_stats(initial_weight_matrix, final_weight_matrix, parameter_name):
    """Quantify how much the spatial_conv weights changed during training."""
    if initial_weight_matrix is None or final_weight_matrix is None:
        return None
    w0 = np.asarray(initial_weight_matrix, dtype=float)
    w1 = np.asarray(final_weight_matrix, dtype=float)
    if w0.shape != w1.shape:
        return {
            "available": False,
            "parameter_name": str(parameter_name),
            "reason": f"Initial/final spatial weights have different shapes: {w0.shape} vs {w1.shape}",
        }
    delta = w1 - w0
    init_l2 = float(np.linalg.norm(w0))
    final_l2 = float(np.linalg.norm(w1))
    delta_l2 = float(np.linalg.norm(delta))
    max_abs_delta = float(np.max(np.abs(delta))) if delta.size else 0.0
    relative_delta = delta_l2 / (init_l2 + 1e-12)
    return {
        "available": True,
        "parameter_name": str(parameter_name),
        "initial_shape": list(w0.shape),
        "final_shape": list(w1.shape),
        "initial_l2": _json_safe_float(init_l2),
        "final_l2": _json_safe_float(final_l2),
        "delta_l2": _json_safe_float(delta_l2),
        "delta_max_abs": _json_safe_float(max_abs_delta),
        "relative_delta": _json_safe_float(relative_delta),
        "changed": bool(delta_l2 > 1e-10),
    }

def extract_spatial_conv_summary(model_or_clf, model_name, ch_names):
    """Extract fold-level PreLocal spatial_conv weights and channel-importance summaries."""
    if model_name != "SignalJEPA_PreLocal":
        return None

    param_name, weight_tensor = _find_first_spatial_weight(model_or_clf)
    if weight_tensor is None:
        return {
            "available": False,
            "reason": "No spatial_conv weight found on model.",
        }

    n_chans = len(ch_names)
    try:
        weight_matrix = _spatial_weight_to_channel_matrix(weight_tensor, n_chans=n_chans)
    except Exception as exc:
        return {
            "available": False,
            "reason": str(exc),
            "parameter_name": param_name,
            "raw_weight_shape": list(weight_tensor.shape),
        }

    # Normalize each virtual spatial filter for easier cross-fold visualization.
    denom = np.max(np.abs(weight_matrix), axis=1, keepdims=True)
    denom[denom == 0] = 1.0
    weight_matrix_norm = weight_matrix / denom

    channel_abs_mean = np.mean(np.abs(weight_matrix), axis=0)
    channel_signed_mean = np.mean(weight_matrix, axis=0)
    channel_l2 = np.sqrt(np.mean(weight_matrix ** 2, axis=0))

    top_order = np.argsort(channel_abs_mean)[::-1]
    top_channels_by_abs_weight = [
        {
            "rank": int(rank + 1),
            "channel_index": int(ch_idx),
            "channel_name": str(ch_names[ch_idx]),
            "abs_mean_weight": _json_safe_float(channel_abs_mean[ch_idx]),
            "signed_mean_weight": _json_safe_float(channel_signed_mean[ch_idx]),
            "l2_weight": _json_safe_float(channel_l2[ch_idx]),
        }
        for rank, ch_idx in enumerate(top_order[:min(15, len(top_order))])
    ]

    return {
        "available": True,
        "parameter_name": param_name,
        "raw_weight_shape": list(weight_tensor.shape),
        "weight_matrix_shape": list(weight_matrix.shape),
        "n_virtual_filters": int(weight_matrix.shape[0]),
        "n_input_channels": int(weight_matrix.shape[1]),
        "channel_names": list(ch_names),
        "weight_matrix": _json_safe_float_list(weight_matrix),
        "weight_matrix_norm": _json_safe_float_list(weight_matrix_norm),
        "channel_abs_mean": _json_safe_float_list(channel_abs_mean),
        "channel_signed_mean": _json_safe_float_list(channel_signed_mean),
        "channel_l2": _json_safe_float_list(channel_l2),
        "top_channels_by_abs_weight": top_channels_by_abs_weight,
    }

def summarize_spatial_conv_for_log(spatial_summary, top_k=8):
    if spatial_summary is None:
        return "spatial_conv=N/A"
    if not spatial_summary.get("available", False):
        return f"spatial_conv=unavailable ({spatial_summary.get('reason')})"
    ch_names = spatial_summary["channel_names"]
    scores = np.asarray(spatial_summary["channel_abs_mean"], dtype=float)
    order = np.argsort(scores)[::-1][:min(top_k, len(scores))]
    top_channels = [(ch_names[i], float(scores[i])) for i in order]
    return f"spatial_conv={spatial_summary['weight_matrix_shape']} top_abs_channels={top_channels}"

def summarize_spatial_update_for_log(update_stats):
    if not update_stats:
        return "spatial_update=N/A"
    if not update_stats.get("available", False):
        return f"spatial_update=unavailable ({update_stats.get('reason')})"
    return (
        "spatial_update="
        f"delta_l2={update_stats['delta_l2']:.8f} "
        f"max_abs={update_stats['delta_max_abs']:.8f} "
        f"relative={update_stats['relative_delta']:.8f} "
        f"changed={update_stats['changed']}"
    )

def _make_eeg_info_for_topomap(ch_names):
    """Use loaded channel info when possible; fallback to a standard montage."""
    try:
        info = mne.pick_info(EEG_INFO.copy(), mne.pick_types(EEG_INFO, eeg=True, meg=False, stim=False))
        # Keep only requested channels in this exact order if possible.
        picks = [info.ch_names.index(ch) for ch in ch_names if ch in info.ch_names]
        if len(picks) == len(ch_names):
            info = mne.pick_info(info, picks)
            return info
    except Exception:
        pass

    # Fallback: create info and attach a broad standard montage.
    info = mne.create_info(ch_names=ch_names, sfreq=float(CONFIG["sfreq"]), ch_types="eeg")
    for montage_name in ("standard_1020", "standard_1005"):
        try:
            montage = mne.channels.make_standard_montage(montage_name)
            info.set_montage(montage, match_case=False, on_missing="ignore")
            return info
        except Exception:
            continue
    return info

def _safe_plot_topomap(values, info, axes, cmap="RdBu_r", vlim=None, names=None, contours=0):
    """
    Compatibility wrapper for different MNE versions.
    Some versions reject show_names; some older versions use vmin/vmax instead of vlim.
    """
    base_kwargs = {
        "data": values,
        "pos": info,
        "axes": axes,
        "show": False,
        "cmap": cmap,
        "contours": contours,
    }
    if names is not None:
        base_kwargs["names"] = names
    if vlim is not None:
        base_kwargs["vlim"] = vlim

    attempts = [dict(base_kwargs)]

    # Without names, in case names are not supported or channel-name display is problematic.
    no_names = dict(base_kwargs)
    no_names.pop("names", None)
    attempts.append(no_names)

    # Older MNE fallback: vmin/vmax instead of vlim.
    if vlim is not None:
        vmin, vmax = vlim
        for kwargs in list(attempts):
            old_kwargs = dict(kwargs)
            old_kwargs.pop("vlim", None)
            old_kwargs["vmin"] = vmin
            old_kwargs["vmax"] = vmax
            attempts.append(old_kwargs)

    last_exc = None
    for kwargs in attempts:
        try:
            return mne.viz.plot_topomap(**kwargs)
        except TypeError as exc:
            last_exc = exc
            continue
    raise last_exc # type: ignore

def plot_spatial_topomap(values, ch_names, title, out_path, cmap="RdBu_r"):
    """Save one EEG topomap. Returns True if successful, False otherwise."""
    values = np.asarray(values, dtype=float)
    info = _make_eeg_info_for_topomap(ch_names)
    try:
        fig, ax = plt.subplots(figsize=(5.5, 4.8))
        vmax = float(np.nanmax(np.abs(values))) if np.any(np.isfinite(values)) else 1.0
        if vmax == 0:
            vmax = 1.0
        _safe_plot_topomap(
            values,
            info,
            axes=ax,
            cmap=cmap,
            vlim=(-vmax, vmax),
            names=ch_names,
            contours=0,
        )
        ax.set_title(title)
        fig.tight_layout()
        fig.savefig(out_path, dpi=int(CONFIG.get("topomap_dpi", 160)))
        plt.close(fig)
        return True
    except Exception as exc:
        print(f"WARNING: Could not save topomap {out_path}: {exc}")
        plt.close("all")
        return False

def plot_spatial_filter_grid(weight_matrix, ch_names, title, out_path, max_filters=8):
    """Save one grid of topomaps, one per virtual spatial filter."""
    weight_matrix = np.asarray(weight_matrix, dtype=float)
    n_filters = min(int(max_filters), weight_matrix.shape[0])
    if n_filters <= 0:
        return False

    info = _make_eeg_info_for_topomap(ch_names)
    n_cols = min(4, n_filters)
    n_rows = int(math.ceil(n_filters / n_cols))
    try:
        fig, axes = plt.subplots(n_rows, n_cols, figsize=(4.0 * n_cols, 3.6 * n_rows))
        axes = np.asarray(axes).reshape(-1)
        vmax = float(np.nanmax(np.abs(weight_matrix[:n_filters]))) if np.any(np.isfinite(weight_matrix[:n_filters])) else 1.0
        if vmax == 0:
            vmax = 1.0
        for i in range(n_filters):
            ax = axes[i]
            _safe_plot_topomap(
                weight_matrix[i],
                info,
                axes=ax,
                cmap="RdBu_r",
                vlim=(-vmax, vmax),
                contours=0,
            )
            ax.set_title(f"virtual filter {i}")
        for j in range(n_filters, len(axes)):
            axes[j].axis("off")
        fig.suptitle(title)
        fig.tight_layout()
        fig.savefig(out_path, dpi=int(CONFIG.get("topomap_dpi", 160)))
        plt.close(fig)
        return True
    except Exception as exc:
        print(f"WARNING: Could not save filter grid {out_path}: {exc}")
        plt.close("all")
        return False


# 5. Training


## 5.1 Build Classifier

In [ ]:
def get_targets(dataset):
    return np.asarray([int(dataset[i][1]) for i in range(len(dataset))], dtype=np.int64)

def make_train_split():
    val_split = CONFIG["val_split"]
    if val_split is None or float(val_split) <= 0.0:
        return None
    return ValidSplit(cv=float(val_split), stratified=True, random_state=12) # type: ignore

def make_callbacks():
    train_split = make_train_split()
    patience = CONFIG["early_stopping_patience"]
    if train_split is None or patience is None or int(patience) <= 0:
        return []

    return [
        (
            "early_stopping",
            EarlyStopping(
                monitor="valid_loss",
                patience=int(patience),
                lower_is_better=True,
                load_best=True,
            ),
        )
    ]

def build_classifier(model, callbacks, max_epochs, fold_seed=None, warm_start=False):
    train_generator = None
    if fold_seed is not None:
        train_generator = torch.Generator()
        train_generator.manual_seed(fold_seed)
    clf_kwargs = {
        "batch_size": CONFIG["batch_size"],
        "max_epochs": int(max_epochs),
        "device": DEVICE,
        "callbacks": callbacks,
        "train_split": make_train_split(),
        "classes": range(TARGET_N_CLASSES),
        "iterator_train__shuffle": True,
        "iterator_train__num_workers": 0,
        "iterator_valid__num_workers": 0,
        "optimizer": torch.optim.Adam,
        "warm_start": warm_start,
    }
    if CONFIG["learning_rate"] is not None:
        clf_kwargs["lr"] = CONFIG["learning_rate"]
    if train_generator is not None:
        clf_kwargs["iterator_train__generator"] = train_generator
    return EEGClassifier(model, **clf_kwargs)

def run_one_batch_finite_sanity_check(model, train_set, model_name):
    if len(train_set) == 0:
        raise RuntimeError(f"{model_name}: train_set is empty during sanity check.")
    batch_size = int(min(CONFIG["batch_size"], len(train_set)))
    sanity_loader = torch.utils.data.DataLoader(train_set, batch_size=batch_size, shuffle=False, num_workers=0)
    batch = next(iter(sanity_loader))
    x_batch = torch.as_tensor(batch[0]).float().to(DEVICE)
    y_batch = torch.as_tensor(batch[1]).long().to(DEVICE)
    was_training = model.training
    model = model.to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(x_batch)
        if not torch.isfinite(logits).all():
            raise RuntimeError(f"{model_name}: non-finite logits detected.")
        loss = torch.nn.functional.cross_entropy(logits, y_batch)
        if not torch.isfinite(loss):
            raise RuntimeError(f"{model_name}: non-finite loss detected.")
    if was_training:
        model.train()
    print(f"    Sanity check passed: finite logits/loss on one training batch for {model_name}.")

def extract_binary_score_vector(score_output, expected_n_samples):
    if score_output is None:
        return None
    scores = np.asarray(score_output)
    if scores.ndim == 1:
        score_vec = scores.astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 2:
        score_vec = scores[:, 1].astype(float)
    elif scores.ndim == 2 and scores.shape[1] == 1:
        score_vec = scores[:, 0].astype(float)
    else:
        return None
    if score_vec.shape[0] != int(expected_n_samples) or not np.isfinite(score_vec).all():
        return None
    return score_vec

def compute_classification_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).astype(int).reshape(-1)
    y_pred = np.asarray(y_pred).astype(int).reshape(-1)
    metrics = {"accuracy": None, "balanced_accuracy": None}
    metrics["accuracy"] = float(accuracy_score(y_true, y_pred)) # type: ignore
    metrics["balanced_accuracy"] = float(balanced_accuracy_score(y_true, y_pred)) # type: ignore
    return metrics

def compute_prediction_probability_diagnostics(clf, test_set, y_pred, n_classes):
    """Optional diagnostics to identify overconfident collapse or low-entropy predictions."""
    if not bool(CONFIG.get("log_probability_diagnostics", True)):
        return None
    try:
        probs = np.asarray(clf.predict_proba(test_set), dtype=float)
    except Exception as exc:
        return {"available": False, "reason": f"predict_proba failed: {exc}"}

    if probs.ndim != 2 or probs.shape[0] != len(test_set):
        return {"available": False, "reason": f"Unexpected probability shape: {list(probs.shape)}"}
    if not np.isfinite(probs).all():
        return {"available": False, "reason": "Non-finite probabilities."}

    # Some classifiers return logits-like values; normalize only if rows do not look like probabilities.
    row_sums = probs.sum(axis=1, keepdims=True)
    if np.any(probs < 0) or not np.allclose(row_sums, 1.0, atol=1e-3):
        exp_probs = np.exp(probs - probs.max(axis=1, keepdims=True))
        probs = exp_probs / np.maximum(exp_probs.sum(axis=1, keepdims=True), 1e-12)

    eps = 1e-12
    confidence = probs.max(axis=1)
    entropy = -np.sum(probs * np.log(probs + eps), axis=1)
    normalized_entropy = entropy / np.log(max(probs.shape[1], 2))
    predicted_class_probability = probs[np.arange(len(probs)), np.asarray(y_pred, dtype=int)]

    return {
        "available": True,
        "probability_shape": list(probs.shape),
        "mean_probability_by_class": _json_safe_float_list(probs.mean(axis=0)),
        "std_probability_by_class": _json_safe_float_list(probs.std(axis=0)),
        "mean_confidence": _json_safe_float(confidence.mean()),
        "std_confidence": _json_safe_float(confidence.std()),
        "mean_prediction_entropy": _json_safe_float(entropy.mean()),
        "mean_normalized_prediction_entropy": _json_safe_float(normalized_entropy.mean()),
        "mean_predicted_class_probability": _json_safe_float(predicted_class_probability.mean()),
    }

def compute_collapse_diagnostics(y_pred, n_classes):
    pred_hist = np.bincount(np.asarray(y_pred, dtype=int), minlength=n_classes)
    n_pred = int(pred_hist.sum())
    if n_pred == 0:
        collapse_ratio = 0.0
        majority_class = None
    else:
        collapse_ratio = float(pred_hist.max() / n_pred)
        majority_class = int(pred_hist.argmax())
    threshold = float(CONFIG.get("collapse_threshold", 0.90))
    return {
        "prediction_histogram": pred_hist.tolist(),
        "collapse_ratio": _json_safe_float(collapse_ratio),
        "collapse_threshold": threshold,
        "collapse_flag": bool(collapse_ratio >= threshold),
        "majority_predicted_class": majority_class,
    }

def run_training_and_eval(train_set, test_set, fold_id, fold_label, n_total_folds=None):
    global PRETRAINED_CHECKPOINT_INFO
    if CONFIG["set_seed"]:
        fold_seed = BASE_SEED
        seed_everything(fold_seed)  # type: ignore
    else:
        fold_seed = None

    y_train = get_targets(train_set)
    y_test = get_targets(test_set)
    train_counts = np.bincount(y_train, minlength=TARGET_N_CLASSES)
    test_counts = np.bincount(y_test, minlength=TARGET_N_CLASSES)

    model_name = CONFIG["model_name"]
    strategy = CONFIG["strategy"]
    warmup_epochs = int(CONFIG["warmup_epochs"])
    model, pretrained_load_summary = initialize_model(model_name)
    PRETRAINED_CHECKPOINT_INFO = dict(pretrained_load_summary)

    total_folds_text = f"/{n_total_folds}" if n_total_folds is not None else ""
    print(f"\nFold {fold_id}{total_folds_text} | {fold_label}")
    print(f"    Train windows:           {len(train_set)} | counts={train_counts.tolist()}")
    print(f"    Test windows:            {len(test_set)} | counts={test_counts.tolist()}")
    print(f"    Downstream model:        {model_name}")
    print(f"    Fine-tune strategy:      {strategy}")
    print(f"    Pretrained loading path: {pretrained_load_summary['loading_path']}")
    print(f"    Pretrained repo:         {pretrained_load_summary.get('repo_id')}")

    initial_spatial_weight_matrix = None
    initial_spatial_param_name = None
    initial_spatial_summary = None
    if bool(CONFIG.get("extract_spatial_conv_weights", False)) and model_name == "SignalJEPA_PreLocal":
        initial_spatial_summary = extract_spatial_conv_summary(model, model_name=model_name, ch_names=CH_NAMES)
        try:
            initial_spatial_weight_matrix, initial_spatial_param_name = get_spatial_conv_weight_matrix(model, CH_NAMES)
        except Exception as exc:
            print(f"    WARNING: Could not snapshot initial spatial_conv weights: {exc}")

    if strategy == "new":
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "new")
        assert_expected_trainable_scope(phase_1_summary, model_name, "new")
        print("    Phase 1 (new):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=make_callbacks(), max_epochs=int(CONFIG["n_epochs"]), fold_seed=fold_seed, warm_start=False)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": None}
        clf.fit(train_set, y=y_train)
    elif strategy == "full":
        if warmup_epochs < 1:
            raise ValueError("For strategy='full', warmup_epochs must be >= 1.")
        phase_1_summary = set_trainable_params_for_phase(model, model_name, "warmup")
        assert_expected_trainable_scope(phase_1_summary, model_name, "warmup")
        print("    Phase 1 (warmup):")
        describe_trainable_params(phase_1_summary)
        run_one_batch_finite_sanity_check(model, train_set, model_name)
        clf = build_classifier(model, callbacks=[], max_epochs=warmup_epochs, fold_seed=fold_seed, warm_start=True)
        clf.fit(train_set, y=y_train)
        phase_2_summary = set_trainable_params_for_phase(clf.module_, model_name, "full")
        print("    Phase 2 (full):")
        describe_trainable_params(phase_2_summary)
        clf.initialize_optimizer()
        remaining_epochs = int(CONFIG["n_epochs"]) - warmup_epochs
        if remaining_epochs < 1:
            raise ValueError("CONFIG['n_epochs'] must be greater than warmup_epochs for strategy='full'.")
        clf.set_params(callbacks=make_callbacks(), max_epochs=remaining_epochs)
        clf.fit(train_set, y=y_train)
        phase_summaries = {"phase_1": phase_1_summary, "phase_2": phase_2_summary}
    else:
        raise ValueError("CONFIG['strategy'] must be 'new' or 'full'.")

    y_pred = clf.predict(test_set)
    metrics = compute_classification_metrics(y_test, y_pred)
    collapse_diagnostics = compute_collapse_diagnostics(y_pred, TARGET_N_CLASSES)
    probability_diagnostics = compute_prediction_probability_diagnostics(clf, test_set, y_pred, TARGET_N_CLASSES)

    spatial_summary = None
    spatial_update_stats = None
    if bool(CONFIG.get("extract_spatial_conv_weights", False)):
        spatial_summary = extract_spatial_conv_summary(clf.module_, model_name=model_name, ch_names=CH_NAMES)
        if (
            bool(CONFIG.get("log_spatial_update_stats", True))
            and initial_spatial_weight_matrix is not None
            and model_name == "SignalJEPA_PreLocal"
        ):
            try:
                final_spatial_weight_matrix, final_param_name = get_spatial_conv_weight_matrix(clf.module_, CH_NAMES)
                spatial_update_stats = compute_spatial_update_stats(
                    initial_weight_matrix=initial_spatial_weight_matrix,
                    final_weight_matrix=final_spatial_weight_matrix,
                    parameter_name=final_param_name or initial_spatial_param_name,
                )
            except Exception as exc:
                spatial_update_stats = {
                    "available": False,
                    "parameter_name": initial_spatial_param_name,
                    "reason": str(exc),
                }

    final_trainable_parameters = summarize_trainable_parameters(clf.module_)
    n_trainable_params_final = count_trainable_from_rows(final_trainable_parameters)

    stopped_epoch = int(clf.history[-1]["epoch"]) if len(clf.history) > 0 else 0  # type: ignore
    valid_loss_curve = [(int(row["epoch"]), float(row["valid_loss"])) for row in clf.history if "valid_loss" in row]
    if valid_loss_curve:
        best_epoch, best_valid_loss = min(valid_loss_curve, key=lambda x: x[1])
    else:
        best_epoch, best_valid_loss = None, None

    cm = confusion_matrix(y_test, y_pred, labels=np.arange(TARGET_N_CLASSES)).tolist()
    pred_hist = collapse_diagnostics["prediction_histogram"]
    acc = metrics["accuracy"]
    bal_acc = metrics["balanced_accuracy"]
    print(
        f"    Result | best_epoch={best_epoch} | stop={stopped_epoch} | "
        f"acc={acc:.4f} | bal_acc={bal_acc:.4f} | pred_hist={pred_hist} | "
        f"collapse_ratio={collapse_diagnostics['collapse_ratio']:.4f} | "
        f"collapse={collapse_diagnostics['collapse_flag']}"
    )
    if spatial_summary is not None:
        print("    " + summarize_spatial_conv_for_log(spatial_summary))
    if spatial_update_stats is not None:
        print("    " + summarize_spatial_update_for_log(spatial_update_stats))

    return {
        "fold_id": int(fold_id),
        "fold_label": str(fold_label),
        "model_name": model_name,
        "strategy": strategy,
        "warmup_epochs": warmup_epochs,
        "n_train": len(train_set),
        "n_test": len(test_set),
        "train_class_counts": train_counts.tolist(),
        "test_class_counts": test_counts.tolist(),
        "pretrained_load": pretrained_load_summary,
        "phase_1_trainable_groups": phase_summaries["phase_1"]["trainable_groups"],
        "phase_1_trainable_params": phase_summaries["phase_1"]["trainable_params"],
        "phase_1_trainable_names": phase_summaries["phase_1"]["trainable_names"],
        "phase_2_trainable_groups": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_groups"],
        "phase_2_trainable_params": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_params"],
        "phase_2_trainable_names": None if phase_summaries["phase_2"] is None else phase_summaries["phase_2"]["trainable_names"],
        "final_trainable_parameters": final_trainable_parameters,
        "n_trainable_params_final": n_trainable_params_final,
        "best_epoch": best_epoch,
        "stopped_epoch": stopped_epoch,
        "best_valid_loss": best_valid_loss,
        "accuracy": acc,
        "balanced_accuracy": bal_acc,
        "confusion_matrix": cm,
        "prediction_histogram": pred_hist,
        "collapse_diagnostics": collapse_diagnostics,
        "probability_diagnostics": probability_diagnostics,
        "spatial_conv_initial": initial_spatial_summary,
        "spatial_conv": spatial_summary,
        "spatial_update_stats": spatial_update_stats,
    }


## 5.2 Subject Cross-Validation Runner

In [ ]:
def make_fold_splits(y, n_folds, n_classes):
    indices = np.arange(len(y))
    counts = np.bincount(y, minlength=n_classes)
    min_class_count = counts.min()
    if min_class_count < n_folds:
        raise ValueError(f"Cannot use {n_folds} folds with class counts={counts.tolist()}. Reduce cv_folds or filter subjects.")
    skf = StratifiedKFold(n_splits=n_folds, shuffle=True, random_state=12)
    folds = []
    for fold_id, (train_idx, test_idx) in enumerate(skf.split(indices, y), start=1):
        folds.append({"fold_id": fold_id, "idx_train": train_idx, "idx_test": test_idx})
    return folds

def run_subject_cv(subject_id, subject_dataset, n_classes, cv_folds):
    y = get_targets(subject_dataset)
    counts = np.bincount(y, minlength=n_classes)
    print(f"\nSubject {subject_id}: {len(subject_dataset)} windows | class_counts={counts.tolist()}")
    folds = make_fold_splits(y, n_folds=cv_folds, n_classes=n_classes)
    results = []
    for fold in folds:
        train_set = Subset(subject_dataset, fold["idx_train"].tolist())
        test_set = Subset(subject_dataset, fold["idx_test"].tolist())
        fold_label = f"subject={subject_id}"
        result = run_training_and_eval(train_set, test_set, fold["fold_id"], fold_label, n_total_folds=cv_folds)
        result["subject_id"] = str(subject_id)
        results.append(result)

    acc_values = [r["accuracy"] for r in results if r["accuracy"] is not None]
    bal_acc_values = [r["balanced_accuracy"] for r in results if r["balanced_accuracy"] is not None]
    print(f"  Subject {subject_id} summary: acc={np.mean(acc_values):.4f}±{np.std(acc_values):.4f}  bal_acc={np.mean(bal_acc_values):.4f}±{np.std(bal_acc_values):.4f}")
    return results

## 5.3 Run All Subjects

In [ ]:
print("=" * 70)
print("STARTING WITHIN-SUBJECT CROSS-VALIDATION")
print("=" * 70)
print(f"Dataset:       {CONFIG['dataset_name']}")
print(f"Subjects:      {list(SUBJECT_WINDOWS.keys())}")
print(f"Model:         {CONFIG['model_name']}")
print(f"Pretrained:    {CONFIG['pretrained_mode']}")
print(f"Strategy:      {CONFIG['strategy']}")
print(f"CV folds:      {CONFIG['cv_folds']}")
print(f"Val split:     {CONFIG['val_split']}")
print(f"Max epochs:    {CONFIG['n_epochs']}")
print(f"Device:        {DEVICE}")
print("=" * 70)

FOLD_RESULTS = []
for sid, subject_ds in SUBJECT_WINDOWS.items():
    FOLD_RESULTS.extend(run_subject_cv(sid, subject_ds, TARGET_N_CLASSES, CONFIG["cv_folds"]))
print(f"\nTotal folds completed: {len(FOLD_RESULTS)}")

# 6. Results


## 6.1 Aggregate Metrics

In [ ]:
def aggregate_results(fold_results):
    # Per-subject aggregation when subject_id exists; per-heldout aggregation for LOSO.
    group_key = "subject_id" if any("subject_id" in r for r in fold_results) else "held_out_subject_id"
    grouped = {}
    for result in fold_results:
        gid = result.get(group_key, "global")
        grouped.setdefault(gid, {"accuracies": [], "balanced_accuracies": []})
        grouped[gid]["accuracies"].append(result.get("accuracy"))
        grouped[gid]["balanced_accuracies"].append(result.get("balanced_accuracy"))
    for gid, m in grouped.items():
        acc_values = [v for v in m["accuracies"] if v is not None]
        bal_values = [v for v in m["balanced_accuracies"] if v is not None]
        m["mean_accuracy"] = float(np.mean(acc_values)) if acc_values else None
        m["std_accuracy"] = float(np.std(acc_values)) if acc_values else None
        m["mean_balanced_accuracy"] = float(np.mean(bal_values)) if bal_values else None
        m["std_balanced_accuracy"] = float(np.std(bal_values)) if bal_values else None

    all_accs = [r["accuracy"] for r in fold_results if r.get("accuracy") is not None]
    all_bals = [r["balanced_accuracy"] for r in fold_results if r.get("balanced_accuracy") is not None]
    global_metrics = {
        "mean_accuracy": float(np.mean(all_accs)) if all_accs else None,
        "std_accuracy": float(np.std(all_accs)) if all_accs else None,
        "mean_balanced_accuracy": float(np.mean(all_bals)) if all_bals else None,
        "std_balanced_accuracy": float(np.std(all_bals)) if all_bals else None,
        "n_groups": len(grouped),
        "n_folds_total": len(fold_results),
    }
    return grouped, global_metrics

SUBJECT_METRICS, GLOBAL_METRICS = aggregate_results(FOLD_RESULTS)

print("=" * 70)
print("AGGREGATED RESULTS")
print("=" * 70)
for gid, m in sorted(SUBJECT_METRICS.items(), key=lambda x: _sort_subject_key(x[0])):
    acc_str = f"{m['mean_accuracy']:.4f}±{m['std_accuracy']:.4f}" if m["mean_accuracy"] is not None else "N/A"
    bal_str = f"{m['mean_balanced_accuracy']:.4f}±{m['std_balanced_accuracy']:.4f}" if m["mean_balanced_accuracy"] is not None else "N/A"
    print(f"  {gid}: acc={acc_str}  bal_acc={bal_str}")
print("-" * 70)
print(f"  OVERALL: acc={GLOBAL_METRICS['mean_accuracy']:.4f}±{GLOBAL_METRICS['std_accuracy']:.4f}  bal_acc={GLOBAL_METRICS['mean_balanced_accuracy']:.4f}±{GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)


## 6.2 Experiment Summary

In [ ]:
print("\n" + "=" * 70)
print("EXPERIMENT SUMMARY")
print("=" * 70)
print(f"Run ID:                 {RUN_ID}")
print(f"Dataset:                {CONFIG['dataset_name']}")
print(f"Labels kept:            {CONFIG['labels_to_keep']}")
print(f"Model:                  {CONFIG['model_name']}")
print(f"Pretrained mode:        {CONFIG['pretrained_mode']}")
print(f"Strategy:               {CONFIG['strategy']}")
print(f"Window:                 {TARGET_TRIAL_DURATION_S:.3f}s / {WINDOW_SAMPLES} samples")
print(f"Artifacts:              {ARTIFACT_DIR}")
print(f"Mean Accuracy:          {GLOBAL_METRICS['mean_accuracy']:.4f} ± {GLOBAL_METRICS['std_accuracy']:.4f}")
print(f"Mean Balanced Accuracy: {GLOBAL_METRICS['mean_balanced_accuracy']:.4f} ± {GLOBAL_METRICS['std_balanced_accuracy']:.4f}")
print("=" * 70)


## 6.3 Spatial Convolution Weight Analysis

This section aggregates the learned PreLocal `spatial_conv` weights across folds and saves CSV/JSON summaries plus topomap visualizations. It is only meaningful for `SignalJEPA_PreLocal`.


In [ ]:
def collect_spatial_conv_records(fold_results):
    records = []
    for result in fold_results:
        spatial = result.get("spatial_conv")
        if not spatial or not spatial.get("available", False):
            continue
        ch_names = spatial["channel_names"]
        abs_scores = np.asarray(spatial["channel_abs_mean"], dtype=float)
        signed_scores = np.asarray(spatial["channel_signed_mean"], dtype=float)
        l2_scores = np.asarray(spatial["channel_l2"], dtype=float)
        collapse = result.get("collapse_diagnostics") or {}
        update_stats = result.get("spatial_update_stats") or {}
        for ch_idx, ch_name in enumerate(ch_names):
            records.append({
                "dataset_name": CONFIG["dataset_name"],
                "subject_id": result.get("subject_id"),
                "fold_id": result.get("fold_id"),
                "channel_index": ch_idx,
                "channel_name": ch_name,
                "abs_mean_weight": float(abs_scores[ch_idx]),
                "signed_mean_weight": float(signed_scores[ch_idx]),
                "l2_weight": float(l2_scores[ch_idx]),
                "accuracy": result.get("accuracy"),
                "balanced_accuracy": result.get("balanced_accuracy"),
                "prediction_histogram": result.get("prediction_histogram"),
                "collapse_ratio": collapse.get("collapse_ratio"),
                "collapse_flag": collapse.get("collapse_flag"),
                "spatial_delta_l2": update_stats.get("delta_l2"),
                "spatial_delta_max_abs": update_stats.get("delta_max_abs"),
                "spatial_relative_delta": update_stats.get("relative_delta"),
                "spatial_changed": update_stats.get("changed"),
            })
    return records

def collect_fold_diagnostic_records(fold_results):
    rows = []
    for result in fold_results:
        collapse = result.get("collapse_diagnostics") or {}
        update_stats = result.get("spatial_update_stats") or {}
        prob = result.get("probability_diagnostics") or {}
        rows.append({
            "dataset_name": CONFIG["dataset_name"],
            "subject_id": result.get("subject_id"),
            "fold_id": result.get("fold_id"),
            "accuracy": result.get("accuracy"),
            "balanced_accuracy": result.get("balanced_accuracy"),
            "prediction_histogram": result.get("prediction_histogram"),
            "collapse_ratio": collapse.get("collapse_ratio"),
            "collapse_flag": collapse.get("collapse_flag"),
            "majority_predicted_class": collapse.get("majority_predicted_class"),
            "spatial_delta_l2": update_stats.get("delta_l2"),
            "spatial_delta_max_abs": update_stats.get("delta_max_abs"),
            "spatial_relative_delta": update_stats.get("relative_delta"),
            "spatial_changed": update_stats.get("changed"),
            "mean_confidence": prob.get("mean_confidence") if prob.get("available") else None,
            "mean_normalized_prediction_entropy": prob.get("mean_normalized_prediction_entropy") if prob.get("available") else None,
            "n_trainable_params_final": result.get("n_trainable_params_final"),
        })
    return rows

SPATIAL_CONV_RECORDS = collect_spatial_conv_records(FOLD_RESULTS)
FOLD_DIAGNOSTIC_RECORDS = collect_fold_diagnostic_records(FOLD_RESULTS)

if CONFIG["model_name"] != "SignalJEPA_PreLocal":
    print("Spatial convolution analysis skipped: selected model is not SignalJEPA_PreLocal.")
elif not CONFIG.get("extract_spatial_conv_weights", False):
    print("Spatial convolution analysis skipped: CONFIG['extract_spatial_conv_weights'] is False.")
elif len(SPATIAL_CONV_RECORDS) == 0:
    print("Spatial convolution analysis skipped: no available spatial_conv records were extracted.")
else:
    spatial_dir = ARTIFACT_DIR / "spatial_conv_analysis"
    spatial_dir.mkdir(parents=True, exist_ok=True)

    spatial_records_path = spatial_dir / "spatial_channel_importance_long.csv"
    spatial_df = pd.DataFrame(SPATIAL_CONV_RECORDS)
    spatial_df.to_csv(spatial_records_path, index=False)

    fold_diagnostics_path = spatial_dir / "fold_diagnostics.csv"
    fold_diag_df = pd.DataFrame(FOLD_DIAGNOSTIC_RECORDS)
    fold_diag_df.to_csv(fold_diagnostics_path, index=False)

    # Fold-level compact JSON with raw filter matrices and diagnostics.
    fold_spatial_json = []
    for result in FOLD_RESULTS:
        spatial = result.get("spatial_conv")
        if spatial and spatial.get("available", False):
            fold_spatial_json.append({
                "subject_id": result.get("subject_id"),
                "fold_id": result.get("fold_id"),
                "accuracy": result.get("accuracy"),
                "balanced_accuracy": result.get("balanced_accuracy"),
                "prediction_histogram": result.get("prediction_histogram"),
                "collapse_diagnostics": result.get("collapse_diagnostics"),
                "probability_diagnostics": result.get("probability_diagnostics"),
                "spatial_update_stats": result.get("spatial_update_stats"),
                "n_trainable_params_final": result.get("n_trainable_params_final"),
                "spatial_conv": spatial,
            })
    fold_json_path = spatial_dir / "spatial_conv_weights_by_fold.json"
    with open(fold_json_path, "w") as f:
        json.dump(fold_spatial_json, f, indent=2)

    # Global channel summary.
    global_summary_df = (
        spatial_df
        .groupby(["channel_index", "channel_name"], as_index=False)
        .agg(
            mean_abs_weight=("abs_mean_weight", "mean"),
            std_abs_weight=("abs_mean_weight", "std"),
            mean_signed_weight=("signed_mean_weight", "mean"),
            mean_l2_weight=("l2_weight", "mean"),
            n_fold_values=("abs_mean_weight", "count"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            collapse_rate=("collapse_flag", "mean"),
            mean_spatial_relative_delta=("spatial_relative_delta", "mean"),
        )
        .sort_values("mean_abs_weight", ascending=False)
    )
    global_summary_path = spatial_dir / "spatial_channel_importance_global.csv"
    global_summary_df.to_csv(global_summary_path, index=False)

    # Subject-level channel summary.
    subject_summary_df = (
        spatial_df
        .groupby(["subject_id", "channel_index", "channel_name"], as_index=False)
        .agg(
            mean_abs_weight=("abs_mean_weight", "mean"),
            std_abs_weight=("abs_mean_weight", "std"),
            mean_signed_weight=("signed_mean_weight", "mean"),
            mean_l2_weight=("l2_weight", "mean"),
            n_fold_values=("abs_mean_weight", "count"),
            mean_collapse_ratio=("collapse_ratio", "mean"),
            collapse_rate=("collapse_flag", "mean"),
            mean_spatial_relative_delta=("spatial_relative_delta", "mean"),
        )
        .sort_values(["subject_id", "mean_abs_weight"], ascending=[True, False])
    )
    subject_summary_path = spatial_dir / "spatial_channel_importance_by_subject.csv"
    subject_summary_df.to_csv(subject_summary_path, index=False)

    # Overall fold diagnostics summary.
    if len(fold_diag_df) > 0:
        diagnostic_summary = {
            "n_folds": int(len(fold_diag_df)),
            "collapse_rate": float(fold_diag_df["collapse_flag"].fillna(False).mean()),
            "mean_collapse_ratio": float(fold_diag_df["collapse_ratio"].dropna().mean()) if fold_diag_df["collapse_ratio"].notna().any() else None,
            "mean_spatial_delta_l2": float(fold_diag_df["spatial_delta_l2"].dropna().mean()) if fold_diag_df["spatial_delta_l2"].notna().any() else None,
            "mean_spatial_relative_delta": float(fold_diag_df["spatial_relative_delta"].dropna().mean()) if fold_diag_df["spatial_relative_delta"].notna().any() else None,
            "spatial_changed_rate": float(fold_diag_df["spatial_changed"].fillna(False).mean()) if "spatial_changed" in fold_diag_df else None,
            "mean_confidence": float(fold_diag_df["mean_confidence"].dropna().mean()) if fold_diag_df["mean_confidence"].notna().any() else None,
            "mean_normalized_prediction_entropy": float(fold_diag_df["mean_normalized_prediction_entropy"].dropna().mean()) if fold_diag_df["mean_normalized_prediction_entropy"].notna().any() else None,
        }
    else:
        diagnostic_summary = {}
    diagnostic_summary_path = spatial_dir / "diagnostic_summary.json"
    with open(diagnostic_summary_path, "w") as f:
        json.dump(diagnostic_summary, f, indent=2)

    print("\nSpatial convolution analysis saved:")
    print(f"  Long channel CSV:        {spatial_records_path}")
    print(f"  Fold diagnostics CSV:    {fold_diagnostics_path}")
    print(f"  Fold weight JSON:        {fold_json_path}")
    print(f"  Global importance CSV:   {global_summary_path}")
    print(f"  Subject importance CSV:  {subject_summary_path}")
    print(f"  Diagnostic summary JSON: {diagnostic_summary_path}")

    print("\nDiagnostic summary:")
    print(json.dumps(diagnostic_summary, indent=2))

    print("\nTop global channels by mean absolute spatial weight:")
    display(global_summary_df.head(15))

    if bool(CONFIG.get("save_spatial_weight_plots", True)):
        ch_names = list(fold_spatial_json[0]["spatial_conv"]["channel_names"])
        global_values = np.zeros(len(ch_names), dtype=float)
        for _, row in global_summary_df.iterrows():
            global_values[int(row["channel_index"])] = float(row["mean_abs_weight"])

        global_topomap_path = spatial_dir / "global_mean_abs_spatial_weight_topomap.png"
        plot_spatial_topomap(
            global_values,
            ch_names=ch_names,
            title=f"{CONFIG['dataset_name']} | global mean abs spatial_conv weight",
            out_path=global_topomap_path,
            cmap="Reds",
        )
        print(f"  Global topomap:          {global_topomap_path}")

        # Global average normalized filter matrix across folds, useful to inspect virtual filters.
        matrices = []
        for item in fold_spatial_json:
            matrices.append(np.asarray(item["spatial_conv"]["weight_matrix_norm"], dtype=float))
        if matrices:
            min_filters = min(m.shape[0] for m in matrices)
            matrices = [m[:min_filters] for m in matrices]
            global_filter_matrix = np.mean(np.stack(matrices, axis=0), axis=0)
            filter_grid_path = spatial_dir / "global_mean_spatial_filters_topomap_grid.png"
            plot_spatial_filter_grid(
                global_filter_matrix,
                ch_names=ch_names,
                title=f"{CONFIG['dataset_name']} | mean normalized spatial filters",
                out_path=filter_grid_path,
                max_filters=int(CONFIG.get("max_spatial_filters_to_plot", 8)),
            )
            print(f"  Global filter grid:      {filter_grid_path}")

        # One topomap per subject.
        for subject_id in sorted(spatial_df["subject_id"].dropna().unique(), key=_sort_subject_key):
            sub_df = subject_summary_df[subject_summary_df["subject_id"] == subject_id]
            subject_values = np.zeros(len(ch_names), dtype=float)
            for _, row in sub_df.iterrows():
                subject_values[int(row["channel_index"])] = float(row["mean_abs_weight"])
            subject_topomap_path = spatial_dir / f"subject_{subject_id}_mean_abs_spatial_weight_topomap.png"
            plot_spatial_topomap(
                subject_values,
                ch_names=ch_names,
                title=f"{CONFIG['dataset_name']} | subject {subject_id} mean abs spatial_conv weight",
                out_path=subject_topomap_path,
                cmap="Reds",
            )
        print(f"  Subject topomaps saved in: {spatial_dir}")


## 6.4 Save Artifacts


In [ ]:
cv_results_path = ARTIFACT_DIR / "cv_results.json"
with open(cv_results_path, "w") as f:
    json.dump(FOLD_RESULTS, f, indent=2)
subject_metrics_path = ARTIFACT_DIR / "subject_metrics.json"
with open(subject_metrics_path, "w") as f:
    json.dump(SUBJECT_METRICS, f, indent=2)
global_metrics_path = ARTIFACT_DIR / "global_metrics.json"
with open(global_metrics_path, "w") as f:
    json.dump(GLOBAL_METRICS, f, indent=2)

run_metadata = {
    "run_id": RUN_ID,
    "artifact_dir": str(ARTIFACT_DIR),
    "dataset_name": CONFIG["dataset_name"],
    "dataset_source": CONFIG["dataset_source"],
    "preprocessed_concat_dir": str(PREPROCESSED_CONCAT_DIR),
    "labels_to_keep": list(CONFIG["labels_to_keep"]),
    "excluded_subjects": list(CONFIG["exclude_subjects"]),
    "target_window_duration_s": TARGET_TRIAL_DURATION_S,
    "window_samples": WINDOW_SAMPLES,
    "subjects": [str(s) for s in SUBJECTS],
    "model_name": CONFIG["model_name"],
    "pretrained_mode": CONFIG["pretrained_mode"],
    "strategy": CONFIG["strategy"],
    "warmup_epochs": int(CONFIG["warmup_epochs"]),
    "pretrained_checkpoint_info": PRETRAINED_CHECKPOINT_INFO,
    "cv_seed": BASE_SEED,
    "global_metrics": GLOBAL_METRICS,
    "extract_spatial_conv_weights": bool(CONFIG.get("extract_spatial_conv_weights", False)),
    "save_spatial_weight_plots": bool(CONFIG.get("save_spatial_weight_plots", False)),
    "collapse_threshold": float(CONFIG.get("collapse_threshold", 0.90)),
    "log_spatial_update_stats": bool(CONFIG.get("log_spatial_update_stats", True)),
    "log_probability_diagnostics": bool(CONFIG.get("log_probability_diagnostics", True)),
}
run_metadata_path = ARTIFACT_DIR / "run_metadata.json"
with open(run_metadata_path, "w") as f:
    json.dump(run_metadata, f, indent=2)

print(f"CV results saved to:      {cv_results_path}")
print(f"Subject metrics saved to: {subject_metrics_path}")
print(f"Global metrics saved to:  {global_metrics_path}")
print(f"Run metadata saved to:    {run_metadata_path}")
print(f"\nAll artifacts in: {ARTIFACT_DIR}")